# Glacier Detection Training Notebook

This notebook provides a comprehensive workflow for training glacier detection models using the glacial lake dataset. It incorporates the GlaViTU (Glacier-VisionTransformer-U-Net) architecture and other available models for accurate glacier mapping.

## Overview
- **Dataset**: Glacial lake dataset with satellite imagery and ground truth masks
- **Models**: GlaViTU, DeepLabV3Plus, ResUNet
- **Features**: Optical (6 bands), DEM (elevation + slope), SAR, InSAR, Thermal
- **Training**: Multi-scale supervision, data augmentation, focal loss
- **Evaluation**: IoU, precision, recall per region and globally

## 1. Setup and Imports

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pickle
import tensorflow as tf
from tqdm import tqdm
import cv2
from sklearn.metrics import confusion_matrix, classification_report

# Import project modules
import configs.config as config
import utils
import dataloaders
import dataloaders.transformations
import models.mapping
import losses
import metrics
import utils.deeplearning

# Configure GPU settings
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Enable memory growth to avoid allocating all GPU memory at once
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # Set GPU as preferred device
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        print(f"✓ GPU configured: {gpus[0]}")
        print(f"✓ Memory growth enabled")
        
        # Optional: Set memory limit (uncomment if needed)
        # tf.config.experimental.set_memory_limit(gpus[0], 20480)  # 20GB limit
        
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠️  No GPU detected - training will use CPU (very slow)")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Print system information
print(f"\n📋 SYSTEM INFO:")
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {len(tf.config.list_physical_devices('GPU'))}")
for i, gpu in enumerate(tf.config.list_physical_devices('GPU')):
    print(f"  GPU {i}: {gpu}")

# Test GPU computation
if tf.config.list_physical_devices('GPU'):
    with tf.device('/GPU:0'):
        test_tensor = tf.random.normal([1000, 1000])
        result = tf.matmul(test_tensor, test_tensor)
    print(f"✓ GPU computation test successful")
else:
    print("⚠️  GPU computation not available")

⚠️  No GPU detected - training will use CPU (very slow)

📋 SYSTEM INFO:
TensorFlow version: 2.13.1
CUDA available: False
GPU devices: 0
⚠️  GPU computation not available


## 2. Configuration Setup

In [2]:
# Dataset paths - Update these paths to your dataset location
TRAIN_DATASET_PATH = "output/train_dataset.hdf5"  # Update this path
VAL_DATASET_PATH = "output/val_dataset.hdf5"      # Update this path
TEST_DATASET_PATH = "output/test_dataset.hdf5"    # Update this path
PREDICTIONS_DIR = "predictions"                     # Directory for saving predictions

# Training configuration
MODEL_NAME = "glavitu_glacier_detection"
PATCH_SIZE = 384
BATCH_SIZE = 8
N_OUTPUTS = 2  # Background and glacier classes
FEATURES = ["optical", "dem"]  # Can add: "co_pol_sar", "cross_pol_sar", "in_sar", "thermal"
LABELS = "outlines"

# Training hyperparameters
LEARNING_RATE = 5e-4
RESTART_EPOCHS = [10, 30, 70, 150]
DROPOUT = 0.10
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.1

# Create directories
os.makedirs("weights", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Features: {FEATURES}")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

Model: glavitu_glacier_detection
Features: ['optical', 'dem']
Patch size: 384x384
Batch size: 8


## 3. Dataset Exploration

In [3]:
# Check if dataset files exist
def check_dataset_files():
    datasets = {
        "Train": TRAIN_DATASET_PATH,
        "Validation": VAL_DATASET_PATH,
        "Test": TEST_DATASET_PATH
    }
    
    for name, path in datasets.items():
        if os.path.exists(path):
            print(f"✓ {name} dataset found: {path}")
            try:
                with h5py.File(path, "r") as f:
                    print(f"  - Number of tiles: {len(f.keys())}")
                    # Show first tile info
                    first_tile = list(f.keys())[0]
                    tile_data = f[first_tile]
                    print(f"  - Sample tile: {first_tile}")
                    print(f"  - Available features: {list(tile_data.keys())}")
                    if 'optical' in tile_data:
                        print(f"  - Optical shape: {tile_data['optical'].shape}")
                    if 'outlines' in tile_data:
                        print(f"  - Labels shape: {tile_data['outlines'].shape}")
                    print(f"  - Attributes: {dict(tile_data.attrs)}")
            except Exception as e:
                print(f"  - Error reading file: {e}")
        else:
            print(f"✗ {name} dataset not found: {path}")
            print(f"  Please update the path or use the demo dataset from the README")

check_dataset_files()

✗ Train dataset not found: output/train_dataset.hdf5
  Please update the path or use the demo dataset from the README
✗ Validation dataset not found: output/val_dataset.hdf5
  Please update the path or use the demo dataset from the README
✗ Test dataset not found: output/test_dataset.hdf5
  Please update the path or use the demo dataset from the README


## 4. Data Visualization

In [ ]:
def visualize_sample_data(dataset_path, n_samples=4):
    """Visualize sample data from the dataset"""
    if not os.path.exists(dataset_path):
        print(f"Dataset not found: {dataset_path}")
        return
    
    with h5py.File(dataset_path, "r") as dataset:
        tiles = list(dataset.keys())[:n_samples]
        
        fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4*n_samples))
        if n_samples == 1:
            axes = axes.reshape(1, -1)
        
        for i, tile_name in enumerate(tiles):
            tile = dataset[tile_name]
            
            # Get optical data (RGB visualization)
            if 'optical' in tile:
                optical = np.array(tile['optical'])
                # Use bands 2,1,0 for RGB (assuming Landsat band order)
                rgb = optical[:, :, [2, 1, 0]]  # Red, Green, Blue
                rgb = np.clip(rgb * 3, 0, 1)  # Enhance contrast
                axes[i, 0].imshow(rgb)
                axes[i, 0].set_title(f'RGB - {tile_name}')
                axes[i, 0].axis('off')
            
            # Get DEM data
            if 'dem' in tile:
                dem = np.array(tile['dem'])
                elevation = dem[:, :, 0]
                axes[i, 1].imshow(elevation, cmap='terrain')
                axes[i, 1].set_title(f'Elevation - {tile_name}')
                axes[i, 1].axis('off')
            
            # Get ground truth
            if 'outlines' in tile:
                outlines = np.array(tile['outlines'])
                if len(outlines.shape) == 3:
                    outlines = outlines[:, :, 1]  # Glacier class
                axes[i, 2].imshow(outlines, cmap='Blues')
                axes[i, 2].set_title(f'Glacier Mask - {tile_name}')
                axes[i, 2].axis('off')
            
            # Print tile info
            attrs = dict(tile.attrs)
            region = attrs.get('region', 'Unknown')
            print(f"Tile {i+1}: {tile_name} - Region: {region}")
        
        plt.tight_layout()
        plt.show()

# Visualize training data samples
print("Sample training data:")
visualize_sample_data(TRAIN_DATASET_PATH, n_samples=2)

## 5. Data Loaders Setup

In [ ]:
def create_input_shapes(features, patch_size):
    """Create input shapes dictionary based on selected features"""
    shapes = {}
    feature_channels = {
        "optical": 6,      # 6 TOA bands
        "dem": 2,          # elevation + slope
        "co_pol_sar": 1,   # Co-polarized SAR
        "cross_pol_sar": 1, # Cross-polarized SAR
        "in_sar": 1,       # InSAR coherence
        "thermal": 1       # Thermal data
    }
    
    for feature in features:
        if feature in feature_channels:
            shapes[feature] = (patch_size, patch_size, feature_channels[feature])
        else:
            print(f"Warning: Unknown feature '{feature}'")
    
    return shapes

def setup_data_loaders():
    """Setup training and validation data loaders"""
    
    # Check if dataset files exist
    if not os.path.exists(TRAIN_DATASET_PATH):
        print(f"Training dataset not found: {TRAIN_DATASET_PATH}")
        print("Please update the path or download the dataset from the README links")
        return None, None
    
    # Create input shapes
    input_shapes = create_input_shapes(FEATURES, PATCH_SIZE)
    print(f"Input shapes: {input_shapes}")
    
    # Training data loader with augmentation
    train_dataset = h5py.File(TRAIN_DATASET_PATH, "r")
    train_sampler = dataloaders.RandomSampler(
        dataset=train_dataset,
        patch_size=PATCH_SIZE,
        features=FEATURES,
        labels=LABELS
    )
    
    # Training augmentations
    train_transformations = [
        dataloaders.transformations.random_vertical_flip(p=0.5),
        dataloaders.transformations.random_horizontal_flip(p=0.5),
        dataloaders.transformations.random_rotation(p=0.75),
        dataloaders.transformations.crop_and_scale(patch_size=PATCH_SIZE, scale=0.8, p=0.5),
        dataloaders.transformations.add_gaussian_noise(["optical"], [0.01], p=0.3),
        dataloaders.transformations.feature_occlusion(["optical"], max_obstruction_size=50, p=0.2)
    ]
    
    train_plugins = [
        dataloaders.Augmentation(train_transformations),
        dataloaders.AddLabelSmoothing(smoothing=LABEL_SMOOTHING)
    ]
    
    train_dataloader = utils.build_dataloader(
        dataloaders.RandomSampler,
        {
            "dataset": train_dataset,
            "patch_size": PATCH_SIZE,
            "features": FEATURES,
            "labels": LABELS
        },
        train_plugins,
        BATCH_SIZE,
        LABELS,
        len_factor=4
    )
    
    # Validation data loader (no augmentation)
    val_dataloader = None
    if os.path.exists(VAL_DATASET_PATH):
        val_dataset = h5py.File(VAL_DATASET_PATH, "r")
        val_dataloader = utils.build_dataloader(
            dataloaders.ConsecutiveSampler,
            {
                "dataset": val_dataset,
                "patch_size": PATCH_SIZE,
                "features": FEATURES,
                "labels": LABELS
            },
            [],  # No plugins for validation
            BATCH_SIZE,
            LABELS
        )
    
    print(f"Training batches: {len(train_dataloader) if train_dataloader else 0}")
    print(f"Validation batches: {len(val_dataloader) if val_dataloader else 0}")
    
    return train_dataloader, val_dataloader, input_shapes

# Setup data loaders
train_dataloader, val_dataloader, input_shapes = setup_data_loaders()

## 6. Model Architecture Setup

In [ ]:
def create_glavitu_model(input_shapes, n_outputs, dropout=0.1):
    """Create GlaViTU model for glacier detection"""
    model = models.mapping.GlaViTU(
        input_shapes=input_shapes,
        n_outputs=n_outputs,
        use_deepsupervision=True,
        use_location=False,  # Can be enabled for region encoding
        last_activation="softmax",
        dropout=dropout,
        inference_dropout=False,
        name="GlaViTU_Glacier"
    )
    return model

def create_deeplab_model(input_shapes, n_outputs, dropout=0.1):
    """Create DeepLabV3Plus model for glacier detection"""
    model = models.mapping.DeepLabV3Plus(
        input_shapes=input_shapes,
        n_outputs=n_outputs,
        last_activation="softmax",
        dropout=dropout,
        inference_dropout=False,
        name="DeepLabV3Plus_Glacier"
    )
    return model

def setup_model(model_type="glavitu"):
    """Setup and compile the model"""
    if input_shapes is None:
        print("Error: Input shapes not defined. Please run data loader setup first.")
        return None
    
    # Create model
    if model_type.lower() == "glavitu":
        model = create_glavitu_model(input_shapes, N_OUTPUTS, DROPOUT)
        n_ds_branches = 2  # Deep supervision branches
    elif model_type.lower() == "deeplab":
        model = create_deeplab_model(input_shapes, N_OUTPUTS, DROPOUT)
        n_ds_branches = 1
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # Setup loss function
    focal_loss = losses.FocalLoss(gamma=FOCAL_GAMMA)
    if n_ds_branches > 1:
        loss = [
            losses.WeightedLoss(
                [focal_loss],
                weights=[0.5 if i + 1 != n_ds_branches else 1.0]
            )
            for i in range(n_ds_branches)
        ]
    else:
        loss = focal_loss
    
    # Setup metrics
    model_metrics = []
    for branch in range(n_ds_branches):
        branch_metrics = [
            tf.keras.metrics.CategoricalAccuracy(),
            tf.keras.metrics.Precision(class_id=0, name=f"precision_background"),
            tf.keras.metrics.Precision(class_id=1, name=f"precision_glacier"),
            tf.keras.metrics.Recall(class_id=0, name=f"recall_background"),
            tf.keras.metrics.Recall(class_id=1, name=f"recall_glacier"),
            metrics.IoU(class_idx=0, name=f"iou_background"),
            metrics.IoU(class_idx=1, name=f"iou_glacier")
        ]
        model_metrics.append(branch_metrics)
    
    if n_ds_branches == 1:
        model_metrics = model_metrics[0]
    
    # Compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=model_metrics
    )
    
    return model, n_ds_branches

# Create and compile model
print("Setting up GlaViTU model...")
model, n_ds_branches = setup_model("glavitu")

if model is not None:
    print(f"Model created successfully with {n_ds_branches} output branches")
    print(f"Total parameters: {model.count_params():,}")
    
    # Display model summary (first few layers)
    try:
        model.summary()
    except:
        print("Model summary not available (requires data to be passed through)")

## 7. Training Callbacks Setup

In [ ]:
def setup_callbacks(model_name, steps_per_epoch, n_outputs):
    """Setup training callbacks"""
    
    # Learning rate schedule with restarts
    restart_steps = [steps_per_epoch * epochs for epochs in RESTART_EPOCHS]
    lr_scheduler = utils.deeplearning.LRRestartsWithCosineDecay(
        start=LEARNING_RATE,
        restart_steps=restart_steps,
        verbose=1
    )
    
    # Model checkpointing - save best weights
    checkpoint_best_loss = tf.keras.callbacks.ModelCheckpoint(
        os.path.join("weights", f"{model_name}_best_loss.h5"),
        monitor="val_loss" if val_dataloader else "loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
    
    checkpoint_best_iou = tf.keras.callbacks.ModelCheckpoint(
        os.path.join("weights", f"{model_name}_best_iou.h5"),
        monitor=f"val_iou_glacier" if val_dataloader else "iou_glacier",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
    
    # CSV logging
    csv_logger = tf.keras.callbacks.CSVLogger(
        os.path.join("logs", f"{model_name}_training_log.csv"),
        append=True
    )
    
    # Early stopping (optional)
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss" if val_dataloader else "loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    )
    
    # Reduce learning rate on plateau
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss" if val_dataloader else "loss",
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    )
    
    callbacks = [
        lr_scheduler,
        checkpoint_best_loss,
        checkpoint_best_iou,
        csv_logger,
        # early_stopping,  # Uncomment if desired
        # reduce_lr,       # Uncomment if desired
    ]
    
    return callbacks

# Setup callbacks
if train_dataloader is not None:
    steps_per_epoch = len(train_dataloader)
    callbacks = setup_callbacks(MODEL_NAME, steps_per_epoch, N_OUTPUTS)
    print(f"Training callbacks setup complete")
    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"Total epochs planned: {RESTART_EPOCHS[-1]}")
else:
    print("Cannot setup callbacks without data loader")

## 8. Training Loop

In [ ]:
def train_model(model, train_dataloader, val_dataloader, callbacks, epochs):
    """Train the glacier detection model"""
    
    print(f"Starting training for {epochs} epochs...")
    print(f"Training batches: {len(train_dataloader)}")
    if val_dataloader:
        print(f"Validation batches: {len(val_dataloader)}")
    
    # Start training
    history = model.fit(
        train_dataloader,
        epochs=epochs,
        validation_data=val_dataloader,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

# Check if we can start training
if model is not None and train_dataloader is not None:
    print("Ready to start training!")
    print("\nTraining configuration:")
    print(f"- Model: {MODEL_NAME}")
    print(f"- Features: {FEATURES}")
    print(f"- Batch size: {BATCH_SIZE}")
    print(f"- Learning rate: {LEARNING_RATE}")
    print(f"- Epochs: {RESTART_EPOCHS[-1]}")
    print(f"- Dropout: {DROPOUT}")
    print(f"- Focal loss gamma: {FOCAL_GAMMA}")
    
    # Uncomment the next line to start training
    # history = train_model(model, train_dataloader, val_dataloader, callbacks, RESTART_EPOCHS[-1])
    
    print("\nTo start training, uncomment the line above and run this cell.")
    print("Training will take several hours depending on your hardware and dataset size.")
else:
    print("Cannot start training. Please check:")
    print(f"- Model created: {model is not None}")
    print(f"- Training data available: {train_dataloader is not None}")
    print("- Dataset paths are correct")
    print("- Dataset files exist and are readable")

## 9. Training Progress Visualization

In [ ]:
def plot_training_history(history):
    """Plot training history"""
    if history is None:
        print("No training history available. Train the model first.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Training Loss')
    if 'val_loss' in history.history:
        axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Accuracy
    acc_key = 'categorical_accuracy' if 'categorical_accuracy' in history.history else 'accuracy'
    if acc_key in history.history:
        axes[0, 1].plot(history.history[acc_key], label='Training Accuracy')
        val_acc_key = f'val_{acc_key}'
        if val_acc_key in history.history:
            axes[0, 1].plot(history.history[val_acc_key], label='Validation Accuracy')
        axes[0, 1].set_title('Model Accuracy')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # IoU for glacier class
    iou_key = 'iou_glacier'
    if iou_key in history.history:
        axes[1, 0].plot(history.history[iou_key], label='Training IoU (Glacier)')
        val_iou_key = f'val_{iou_key}'
        if val_iou_key in history.history:
            axes[1, 0].plot(history.history[val_iou_key], label='Validation IoU (Glacier)')
        axes[1, 0].set_title('Glacier IoU')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('IoU')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    # Learning rate
    if 'lr' in history.history:
        axes[1, 1].plot(history.history['lr'], label='Learning Rate')
        axes[1, 1].set_title('Learning Rate Schedule')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_yscale('log')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()

# Uncomment after training to visualize results
# plot_training_history(history)

## 10. Model Evaluation

In [ ]:
def evaluate_model(model, test_dataloader, model_name):
    """Evaluate the trained model"""
    if model is None or test_dataloader is None:
        print("Model or test data not available")
        return
    
    print("Evaluating model on test set...")
    
    # Evaluate on test set
    test_results = model.evaluate(test_dataloader, verbose=1)
    
    # Print results
    print("\nTest Results:")
    for i, metric_name in enumerate(model.metrics_names):
        print(f"{metric_name}: {test_results[i]:.4f}")
    
    return test_results

def predict_and_visualize(model, test_dataloader, n_samples=4):
    """Make predictions and visualize results"""
    if model is None or test_dataloader is None:
        print("Model or test data not available")
        return
    
    print("Making predictions...")
    
    # Get a batch of test data
    test_batch = next(iter(test_dataloader))
    x_test, y_test = test_batch
    
    # Make predictions
    predictions = model.predict(x_test)
    
    # Handle deep supervision (multiple outputs)
    if isinstance(predictions, list):
        predictions = predictions[-1]  # Use final output
    
    # Convert to class predictions
    pred_classes = np.argmax(predictions, axis=-1)
    true_classes = np.argmax(y_test, axis=-1)
    
    # Visualize results
    n_samples = min(n_samples, len(x_test))
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_samples):
        # Original image (RGB from optical)
        if 'optical' in x_test:
            optical = x_test['optical'][i]
            rgb = optical[:, :, [2, 1, 0]]  # RGB bands
            rgb = np.clip(rgb * 3, 0, 1)  # Enhance contrast
            axes[i, 0].imshow(rgb)
            axes[i, 0].set_title(f'Input RGB {i+1}')
            axes[i, 0].axis('off')
        
        # Ground truth
        axes[i, 1].imshow(true_classes[i], cmap='Blues', vmin=0, vmax=1)
        axes[i, 1].set_title(f'Ground Truth {i+1}')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(pred_classes[i], cmap='Blues', vmin=0, vmax=1)
        axes[i, 2].set_title(f'Prediction {i+1}')
        axes[i, 2].axis('off')
        
        # Prediction confidence (glacier class)
        confidence = predictions[i, :, :, 1]  # Glacier class probability
        axes[i, 3].imshow(confidence, cmap='Reds', vmin=0, vmax=1)
        axes[i, 3].set_title(f'Confidence {i+1}')
        axes[i, 3].axis('off')
        
        # Calculate IoU for this sample
        intersection = np.sum((pred_classes[i] == 1) & (true_classes[i] == 1))
        union = np.sum((pred_classes[i] == 1) | (true_classes[i] == 1))
        iou = intersection / union if union > 0 else 0
        print(f"Sample {i+1} IoU: {iou:.3f}")
    
    plt.tight_layout()
    plt.show()

# Setup test data loader if available
test_dataloader = None
if os.path.exists(TEST_DATASET_PATH):
    test_dataset = h5py.File(TEST_DATASET_PATH, "r")
    test_dataloader = utils.build_dataloader(
        dataloaders.ConsecutiveSampler,
        {
            "dataset": test_dataset,
            "patch_size": PATCH_SIZE,
            "features": FEATURES,
            "labels": LABELS
        },
        [],  # No plugins for testing
        BATCH_SIZE,
        LABELS
    )
    print(f"Test data loader created with {len(test_dataloader)} batches")
else:
    print(f"Test dataset not found: {TEST_DATASET_PATH}")

# Uncomment after training to evaluate
# evaluate_model(model, test_dataloader, MODEL_NAME)
# predict_and_visualize(model, test_dataloader, n_samples=4)

## 11. Model Saving and Loading

In [ ]:
def save_model(model, model_name):
    """Save the trained model"""
    if model is None:
        print("No model to save")
        return
    
    # Save full model
    model_path = os.path.join("weights", f"{model_name}_full_model.h5")
    model.save(model_path)
    print(f"Full model saved to: {model_path}")
    
    # Save weights only
    weights_path = os.path.join("weights", f"{model_name}_weights.h5")
    model.save_weights(weights_path)
    print(f"Model weights saved to: {weights_path}")

def load_model_weights(model, weights_path):
    """Load pre-trained weights"""
    if not os.path.exists(weights_path):
        print(f"Weights file not found: {weights_path}")
        return False
    
    try:
        model.load_weights(weights_path)
        print(f"Weights loaded from: {weights_path}")
        return True
    except Exception as e:
        print(f"Error loading weights: {e}")
        return False

# Example usage:
# save_model(model, MODEL_NAME)
# load_model_weights(model, "weights/glavitu_glacier_detection_best_iou.h5")

print("Model saving/loading functions ready")
print("Available weight files:")
weights_dir = "weights"
if os.path.exists(weights_dir):
    weight_files = [f for f in os.listdir(weights_dir) if f.endswith('.h5')]
    for f in weight_files:
        print(f"  - {f}")
else:
    print("  No weights directory found")

## 12. Inference on New Data

In [ ]:
def predict_on_new_data(model, features_dict, patch_size=384):
    """Make predictions on new satellite imagery"""
    if model is None:
        print("No model available for prediction")
        return None
    
    # Prepare input data
    height, width = features_dict['optical'].shape[:2]
    
    # Pad to patch size multiples
    target_height = patch_size * int(height // patch_size + 1)
    target_width = patch_size * int(width // patch_size + 1)
    pad_height = (target_height - height) // 2
    pad_width = (target_width - width) // 2
    
    padded_features = {}
    for feature_name, feature_data in features_dict.items():
        if len(feature_data.shape) == 3:
            h, w, c = feature_data.shape
            padded = np.zeros((target_height, target_width, c))
            padded[pad_height:pad_height + h, pad_width:pad_width + w, :] = feature_data
            padded_features[feature_name] = padded
    
    # Create patches for prediction
    patches = []
    positions = []
    
    for y in range(0, target_height, patch_size):
        for x in range(0, target_width, patch_size):
            patch = {}
            for feature_name, feature_data in padded_features.items():
                patch[feature_name] = feature_data[y:y+patch_size, x:x+patch_size, :]
            patches.append(patch)
            positions.append((y, x))
    
    # Convert to batch format
    batch_data = {}
    for feature_name in padded_features.keys():
        batch_data[feature_name] = np.array([patch[feature_name] for patch in patches])
    
    # Make predictions
    print(f"Predicting on {len(patches)} patches...")
    predictions = model.predict(batch_data, verbose=1)
    
    # Handle deep supervision
    if isinstance(predictions, list):
        predictions = predictions[-1]
    
    # Reconstruct full prediction map
    pred_map = np.zeros((target_height, target_width, predictions.shape[-1]))
    
    for i, (y, x) in enumerate(positions):
        pred_map[y:y+patch_size, x:x+patch_size, :] = predictions[i]
    
    # Remove padding
    pred_map = pred_map[pad_height:pad_height + height, pad_width:pad_width + width, :]
    
    return pred_map

# Example usage (uncomment after training):
# Load your own satellite imagery data
# features = {
#     'optical': your_optical_data,  # Shape: (height, width, 6)
#     'dem': your_dem_data,          # Shape: (height, width, 2)
# }
# prediction_map = predict_on_new_data(model, features)
# glacier_probability = prediction_map[:, :, 1]  # Glacier class probability

print("Inference functions ready")
print("To use on new data:")
print("1. Load your satellite imagery (optical + DEM)")
print("2. Normalize using the same statistics as training data")
print("3. Call predict_on_new_data() function")
print("4. Extract glacier probability from output[:, :, 1]")

## 13. Summary and Next Steps

In [ ]:
print("=" * 60)
print("GLACIER DETECTION TRAINING NOTEBOOK SUMMARY")
print("=" * 60)

print("\n📋 SETUP COMPLETED:")
print(f"✓ Model architecture: {MODEL_NAME}")
print(f"✓ Features configured: {FEATURES}")
print(f"✓ Training parameters set")
print(f"✓ Data loaders prepared")
print(f"✓ Callbacks configured")

print("\n🎯 TO START TRAINING:")
print("1. Update dataset paths in Section 2")
print("2. Download datasets from README links if needed")
print("3. Uncomment training line in Section 8")
print("4. Run the training cell")

print("\n📊 AFTER TRAINING:")
print("1. Visualize training history (Section 9)")
print("2. Evaluate on test set (Section 10)")
print("3. Save trained model (Section 11)")
print("4. Use for inference on new data (Section 12)")

print("\n🔧 CUSTOMIZATION OPTIONS:")
print("- Add more features: co_pol_sar, cross_pol_sar, in_sar, thermal")
print("- Enable region encoding: use_location=True")
print("- Try different models: DeepLabV3Plus, ResUNet")
print("- Adjust hyperparameters: learning rate, batch size, dropout")
print("- Modify augmentations in data loader setup")

print("\n📁 OUTPUT FILES:")
print("- weights/: Trained model weights")
print("- logs/: Training logs and metrics")
print("- predictions/: Model predictions on test data")

print("\n⚠️  IMPORTANT NOTES:")
print("- Training requires significant GPU memory (24GB+ recommended)")
print("- Full training takes several hours to days")
print("- Monitor GPU memory usage and reduce batch size if needed")
print("- Use demo dataset for quick testing")

print("\n🚀 READY TO TRAIN GLACIER DETECTION MODELS!")
print("=" * 60)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('training_log.csv')
df.tail(1)

,epoch,accuracy,dice_coefficient,iou_metric,loss,lr,precision,recall,val_accuracy,val_dice_coefficient,val_iou_metric,val_loss,val_precision,val_recall
3,3,0.97749,0.893569,0.847087,0.18504,0.0001,0.899883,0.934896,0.978307,0.900391,0.867004,0.172828,0.905816,0.94018


: 